In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from umap import UMAP
from mpl_toolkits.mplot3d import Axes3D
import plotly.graph_objects as go
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("库加载完成")

/home/ubuntu/.conda/envs/spike_sorting/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


库加载完成


In [26]:
neural_response = np.load("/media/ubuntu/sda/TrippleN/customize/neuron_responses_1000.npy")

In [39]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import cdist

X = np.asarray(neural_response, dtype=np.float32)
if X.shape[0] != 1000 and X.shape[1] == 1000:
    X = X.T
if X.shape[0] != 1000:
    raise ValueError(f"neural_response 期望样本数为 1000，得到 {X.shape}")

pca_resp = PCA(n_components=500, random_state=42)
X_500 = pca_resp.fit_transform(X).astype(np.float32)
print('response PCA:', X.shape, '->', X_500.shape, 'ev=', float(pca_resp.explained_variance_ratio_.sum()))

fc6_path = '/media/ubuntu/sda/TrippleN/customize/decoding_analysis/alexnet_fc6_features_1000.npy'
fc6 = np.load(fc6_path).astype(np.float32)
if fc6.shape != (1000, 4096):
    raise ValueError(f"fc6 期望形状 (1000,4096)，得到 {fc6.shape}")

pca_fc6 = PCA(n_components=500, random_state=42)
fc6_100 = pca_fc6.fit_transform(fc6).astype(np.float32)
print('fc6 PCA:', fc6.shape, '->', fc6_100.shape, 'ev=', float(pca_fc6.explained_variance_ratio_.sum()))

def loocv_ridge_predict(X, Y, alpha=1.0):
    X = np.asarray(X, dtype=np.float32)
    Y = np.asarray(Y, dtype=np.float32)
    n = X.shape[0]
    preds = np.zeros_like(Y)
    for i in range(n):
        if (i + 1) % 100 == 0:
            print(f"LOOCV {i+1}/{n}")
        train_idx = np.concatenate([np.arange(i), np.arange(i + 1, n)])
        X_train = X[train_idx]
        Y_train = Y[train_idx]
        X_test = X[i:i+1]

        scaler = StandardScaler(with_mean=True, with_std=True)
        X_train_s = scaler.fit_transform(X_train)
        X_test_s = scaler.transform(X_test)

        reg = Ridge(alpha=alpha, fit_intercept=True)
        reg.fit(X_train_s, Y_train)
        preds[i] = reg.predict(X_test_s)[0]
    return preds

pred_fc6_100 = loocv_ridge_predict(X_500, fc6_100, alpha=3.0)
print('pred_fc6_100 shape:', pred_fc6_100.shape)

sim = 1 - cdist(pred_fc6_100, fc6_100, metric='correlation')
acc_full = float(np.mean(np.argmax(sim, axis=1) == np.arange(1000)))
print('full 1000-way acc (deterministic):', acc_full)

response PCA: (1000, 15652) -> (1000, 500) ev= 0.8664945363998413
fc6 PCA: (1000, 4096) -> (1000, 500) ev= 0.9105314016342163
LOOCV 100/1000
LOOCV 200/1000
LOOCV 300/1000
LOOCV 400/1000
LOOCV 500/1000
LOOCV 600/1000
LOOCV 700/1000
LOOCV 800/1000
LOOCV 900/1000
LOOCV 1000/1000
pred_fc6_100 shape: (1000, 500)
full 1000-way acc (deterministic): 0.568


In [40]:
import numpy as np

def identification_1000way_random_trials(sim_matrix, n_trials=50000, n_classes=1000, seed=42):
    rng = np.random.default_rng(seed)
    n = sim_matrix.shape[0]
    if sim_matrix.shape != (n_classes, n_classes):
        raise ValueError(f"期望 sim_matrix 形状 ({n_classes},{n_classes})，得到 {sim_matrix.shape}")

    correct = 0
    for t in range(n_trials):
        i = int(rng.integers(0, n))
        neg = rng.choice(n - 1, size=n_classes - 1, replace=False)
        neg = neg + (neg >= i)
        cand = np.concatenate(([i], neg))
        scores = sim_matrix[i, cand]
        if int(np.argmax(scores)) == 0:
            correct += 1
        if (t + 1) % 10000 == 0:
            print(f"trials {t+1}/{n_trials}  acc={correct/(t+1):.4f}")
    return correct / n_trials

acc_50k = identification_1000way_random_trials(sim, n_trials=50000, n_classes=1000, seed=42)
print('random 1000-way acc over 50000 trials:', float(acc_50k))

trials 10000/50000  acc=0.5632
trials 20000/50000  acc=0.5665
trials 30000/50000  acc=0.5674
trials 40000/50000  acc=0.5653
trials 50000/50000  acc=0.5669
random 1000-way acc over 50000 trials: 0.56688


In [41]:
import os
import pickle
import numpy as np
from scipy.spatial.distance import cdist
from PIL import Image

_ddir_nn = '/media/ubuntu/sda/TrippleN/customize/decoding_analysis'
_stim_dir = '/media/ubuntu/sda/TrippleN/stimuli'
_out_nn = os.path.join(_ddir_nn, 'loocv_per_stim_original_image_nn_fc6_500d.pkl')

_pca_mean_fc6 = np.asarray(pca_fc6.mean_, dtype=np.float32)
_pca_comp_fc6 = np.asarray(pca_fc6.components_, dtype=np.float32)

_pred = np.asarray(pred_fc6_100, dtype=np.float64)
_tgt = np.asarray(fc6_100, dtype=np.float64)

_dmat = cdist(_pred, _tgt, metric='euclidean')
_nn = np.argmin(_dmat, axis=1).astype(np.int32)

_bmp = sorted([x for x in os.listdir(_stim_dir) if x.lower().endswith('.bmp')])[:1000]
if len(_bmp) < 1000:
    raise ValueError('stimuli 不足 1000 张 bmp')

_orig_imgs = []
for _fn in _bmp:
    _orig_imgs.append(np.asarray(Image.open(os.path.join(_stim_dir, _fn)), dtype=np.uint8))
_orig_imgs = np.stack(_orig_imgs, axis=0)

_nn_fc6 = _tgt[_nn].astype(np.float32)
_correct = (_nn == np.arange(1000, dtype=np.int32))

_nn_pack = {
    'model': 'alexnet_fc6',
    'space': 'target_reduced_500d_pca',
    'neighbor_rule': 'argmin_j euclidean(pred[i], target[j])',
    'decode_correct': _correct,
    'image_filenames': _bmp,
    'nn_stimulus_idx': _nn,
    'original_images_uint8': _orig_imgs,
    'nn_alexnet_fc6_500d': _nn_fc6,
    'alexnet_fc6_pca_mean': _pca_mean_fc6,
    'alexnet_fc6_pca_components': _pca_comp_fc6,
}

with open(_out_nn, 'wb') as f:
    pickle.dump(_nn_pack, f, protocol=pickle.HIGHEST_PROTOCOL)

print('saved', _out_nn)
print('original_images_uint8', _orig_imgs.shape, 'nn_alexnet_fc6_500d', _nn_fc6.shape)
print('decode_correct mean', float(np.mean(_correct)), 'n_correct', int(np.sum(_correct)))
print('alexnet_fc6_pca_mean', _pca_mean_fc6.shape, 'alexnet_fc6_pca_components', _pca_comp_fc6.shape)

saved /media/ubuntu/sda/TrippleN/customize/decoding_analysis/loocv_per_stim_original_image_nn_fc6_500d.pkl
original_images_uint8 (1000, 227, 227, 3) nn_alexnet_fc6_500d (1000, 500)
decode_correct mean 0.268 n_correct 268
alexnet_fc6_pca_mean (4096,) alexnet_fc6_pca_components (500, 4096)
